# Lab 05: Tool Description Engineering

**Goal:** Discover how tool description quality directly affects LLM tool selection accuracy.

**What you'll learn:**
- The LLM reads tool descriptions to decide which tool to call
- Vague descriptions lead to wrong tool selections
- Clear, specific descriptions with examples lead to accurate selections
- Descriptions must be unique and non-overlapping to avoid confusion

**How this differs from Lab 02 (ReAct):**
- Lab 02 focused on the *reasoning loop* (Thought -> Action -> Observation)
- This lab focuses on *description quality* as a lever for tool selection accuracy
- Same underlying mechanism, completely different skill: writing good tool specs

| Criteria | Points |
|----------|--------|
| TODO 1 | 4 |
| TODO 2 | 3 |
| TODO 3 | 3 |
| **Total** | **10** |

## Imports and Setup

In [ ]:
import json
import os
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOllama(model="llama3.2:1b")

# Output directory
OUTPUT_DIR = "/tmp/k8s-lab-03-05"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

## Step 1: Define 5 Business-Domain Tools

These tools simulate real enterprise operations. Each is a simple Python function.

| Tool | Purpose | Example Input |
|------|---------|---------------|
| `lookup_employee` | Find employee info by name | `"Alice"` |
| `check_inventory` | Check stock levels for a product | `"Laptop"` |
| `calculate_shipping` | Compute shipping cost by weight | `"2.5"` |
| `translate_text` | Translate text to Hindi (simulated) | `"Hello"` |
| `summarize_text` | Produce a shorter version of text | `"Long paragraph..."` |

In [ ]:
# --- Tool implementations (these stay the same across all experiments) ---

EMPLOYEE_DB = {
    "alice": {"name": "Alice Johnson", "role": "Engineering Manager", "department": "Platform", "location": "Bangalore"},
    "bob": {"name": "Bob Singh", "role": "Senior Developer", "department": "Backend", "location": "Pune"},
    "carol": {"name": "Carol Patel", "role": "Data Scientist", "department": "ML Team", "location": "Hyderabad"},
}

INVENTORY_DB = {
    "laptop": {"product": "Laptop", "stock": 142, "warehouse": "Mumbai"},
    "keyboard": {"product": "Keyboard", "stock": 580, "warehouse": "Delhi"},
    "monitor": {"product": "Monitor", "stock": 37, "warehouse": "Mumbai"},
    "headset": {"product": "Headset", "stock": 0, "warehouse": "Chennai"},
}


def lookup_employee(name: str) -> str:
    """Look up employee information by name."""
    key = name.strip().lower()
    for k, v in EMPLOYEE_DB.items():
        if k in key or key in k:
            return json.dumps(v)
    return f"No employee found matching '{name}'."


def check_inventory(product: str) -> str:
    """Check inventory stock level for a product."""
    key = product.strip().lower()
    for k, v in INVENTORY_DB.items():
        if k in key or key in k:
            if v["stock"] == 0:
                return f"{v['product']}: OUT OF STOCK (warehouse: {v['warehouse']})"
            return f"{v['product']}: {v['stock']} units in stock (warehouse: {v['warehouse']})"
    return f"Product '{product}' not found in inventory."


def calculate_shipping(weight_kg: str) -> str:
    """Calculate shipping cost based on weight in kg."""
    try:
        w = float(weight_kg)
        if w <= 0:
            return "Weight must be positive."
        # Base rate: 50 INR + 30 INR per kg
        cost = 50 + (30 * w)
        return f"Shipping cost for {w} kg: INR {cost:.2f} (base 50 + 30/kg)"
    except ValueError:
        return f"Invalid weight: '{weight_kg}'. Provide a number in kg."


def translate_text(text: str) -> str:
    """Simulate translating text to Hindi."""
    translations = {
        "hello": "Namaste",
        "thank you": "Dhanyavaad",
        "good morning": "Suprabhat",
        "how are you": "Aap kaise hain",
    }
    lower = text.strip().lower()
    if lower in translations:
        return f"'{text}' -> Hindi: '{translations[lower]}'"
    return f"'{text}' -> Hindi: '[simulated translation of: {text}]'"


def summarize_text(text: str) -> str:
    """Produce a shorter version of the input text."""
    words = text.split()
    if len(words) <= 10:
        return f"Summary: {text} (already short)"
    short = " ".join(words[:10])
    return f"Summary ({len(words)} words -> 10): {short}..."


# Map tool names to functions (constant across all experiments)
TOOL_FUNCTIONS = {
    "lookup_employee": lookup_employee,
    "check_inventory": check_inventory,
    "calculate_shipping": calculate_shipping,
    "translate_text": translate_text,
    "summarize_text": summarize_text,
}

print(f"Defined {len(TOOL_FUNCTIONS)} tools: {list(TOOL_FUNCTIONS.keys())}")

## Step 2: System Prompt Builder and Tool Parser

We need a function that builds a system prompt from tool descriptions, and one that parses the LLM's tool choice from its response.

In [ ]:
def build_tool_system_prompt(descriptions: dict) -> str:
    """Build a system prompt listing available tools.
    
    Args:
        descriptions: dict mapping tool_name -> description string
    """
    tool_lines = "\n".join(
        f"- {name}: {desc}" for name, desc in descriptions.items()
    )
    return f"""You are a helpful assistant with access to tools.

Available tools:
{tool_lines}

When the user asks a question, decide which tool to use and respond with EXACTLY this JSON:
{{"tool": "tool_name", "argument": "the argument"}}

Rules:
- You MUST pick exactly one tool for every request.
- Respond ONLY with the JSON object, nothing else.
- Do not add any text before or after the JSON."""


def parse_tool_choice(response_text: str) -> str | None:
    """Extract the tool name the LLM chose from its response."""
    try:
        start = response_text.index("{")
        end = response_text.rindex("}") + 1
        call = json.loads(response_text[start:end])
        return call.get("tool", "").strip()
    except (ValueError, json.JSONDecodeError):
        # Fallback: check if any tool name appears in the response
        for name in TOOL_FUNCTIONS:
            if name in response_text:
                return name
        return None


# Quick test
test_resp = '{"tool": "check_inventory", "argument": "laptop"}'
assert parse_tool_choice(test_resp) == "check_inventory"
print("Parser works correctly.")

## Step 3: Define Test Questions

Each question has an **expected tool**. We will use the same questions for every experiment, only changing the descriptions.

In [ ]:
TEST_QUESTIONS = [
    {"question": "What department does Alice work in?", "expected_tool": "lookup_employee"},
    {"question": "How many laptops do we have in stock?", "expected_tool": "check_inventory"},
    {"question": "How much will it cost to ship a 3.5 kg package?", "expected_tool": "calculate_shipping"},
    {"question": "How do you say 'thank you' in Hindi?", "expected_tool": "translate_text"},
    {"question": "Give me a shorter version of this paragraph: Artificial intelligence is transforming industries worldwide by automating complex tasks and enabling data-driven decisions at scale", "expected_tool": "summarize_text"},
    {"question": "Is Bob a manager or a developer?", "expected_tool": "lookup_employee"},
]

print(f"{len(TEST_QUESTIONS)} test questions defined.")
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"  Q{i}: [{q['expected_tool']}] {q['question'][:60]}...")

## Step 4: Bad Descriptions Experiment

Let's give the tools **vague, unhelpful descriptions** and see how the LLM performs.

In [ ]:
BAD_DESCRIPTIONS = {
    "lookup_employee": "do stuff",
    "check_inventory": "handle data",
    "calculate_shipping": "process numbers",
    "translate_text": "process text",
    "summarize_text": "handle text",
}

bad_prompt = build_tool_system_prompt(BAD_DESCRIPTIONS)
print("=== BAD DESCRIPTIONS SYSTEM PROMPT ===")
print(bad_prompt)
print("\n--- Running test with bad descriptions ---")

bad_correct = 0
bad_results = []

for q in TEST_QUESTIONS:
    response = llm.invoke([
        SystemMessage(content=bad_prompt),
        HumanMessage(content=q["question"]),
    ])
    chosen = parse_tool_choice(response.content)
    is_correct = chosen == q["expected_tool"]
    if is_correct:
        bad_correct += 1
    bad_results.append({
        "question": q["question"],
        "expected": q["expected_tool"],
        "chosen": chosen,
        "correct": is_correct,
    })
    status = "CORRECT" if is_correct else "WRONG"
    print(f"  [{status}] Expected: {q['expected_tool']:20s} | Chosen: {str(chosen):20s} | Q: {q['question'][:50]}")

bad_accuracy = bad_correct / len(TEST_QUESTIONS)
print(f"\nBad descriptions accuracy: {bad_correct}/{len(TEST_QUESTIONS)} = {bad_accuracy:.0%}")

## Step 5: Good Descriptions Experiment

Now let's give the **same tools** clear, specific descriptions with examples.

In [ ]:
GOOD_DESCRIPTIONS = {
    "lookup_employee": "Look up employee information (name, role, department, location) by employee name. Example: argument 'Alice' returns her job title and department.",
    "check_inventory": "Check product stock levels in the warehouse. Returns quantity in stock and warehouse location. Example: argument 'Laptop' returns '142 units in stock'.",
    "calculate_shipping": "Calculate the shipping cost for a package given its weight in kilograms. Returns cost in INR. Example: argument '3.5' returns 'INR 155.00'.",
    "translate_text": "Translate an English word or phrase into Hindi. Example: argument 'Hello' returns 'Namaste'.",
    "summarize_text": "Summarize a long piece of text into a shorter version. Takes the full text as argument and returns an abbreviated form.",
}

good_prompt = build_tool_system_prompt(GOOD_DESCRIPTIONS)
print("=== GOOD DESCRIPTIONS SYSTEM PROMPT ===")
print(good_prompt)
print("\n--- Running test with good descriptions ---")

good_correct = 0
good_results = []

for q in TEST_QUESTIONS:
    response = llm.invoke([
        SystemMessage(content=good_prompt),
        HumanMessage(content=q["question"]),
    ])
    chosen = parse_tool_choice(response.content)
    is_correct = chosen == q["expected_tool"]
    if is_correct:
        good_correct += 1
    good_results.append({
        "question": q["question"],
        "expected": q["expected_tool"],
        "chosen": chosen,
        "correct": is_correct,
    })
    status = "CORRECT" if is_correct else "WRONG"
    print(f"  [{status}] Expected: {q['expected_tool']:20s} | Chosen: {str(chosen):20s} | Q: {q['question'][:50]}")

good_accuracy = good_correct / len(TEST_QUESTIONS)
print(f"\nGood descriptions accuracy: {good_correct}/{len(TEST_QUESTIONS)} = {good_accuracy:.0%}")

## Step 6: Quick Comparison

Let's compare bad vs good descriptions side by side.

In [ ]:
print("=" * 60)
print("DESCRIPTION QUALITY COMPARISON")
print("=" * 60)
print(f"{'Level':<15} {'Correct':>8} {'Total':>6} {'Accuracy':>10}")
print("-" * 45)
print(f"{'Bad':<15} {bad_correct:>8} {len(TEST_QUESTIONS):>6} {bad_accuracy:>10.0%}")
print(f"{'Good':<15} {good_correct:>8} {len(TEST_QUESTIONS):>6} {good_accuracy:>10.0%}")
print("=" * 60)
improvement = good_accuracy - bad_accuracy
print(f"Improvement from good descriptions: +{improvement:.0%}")

---

## TODO 1: Implement `test_tool_selection` Function (4 points)

Refactor the testing logic into a **reusable function** so we can easily run experiments.

**Requirements:**
- Takes `descriptions` (dict of tool_name -> description), `questions` (list of dicts with 'question' and 'expected_tool'), and `llm`
- Builds the system prompt using `build_tool_system_prompt(descriptions)`
- Sends each question to the LLM and parses the tool choice
- Returns a dict with:
  - `"accuracy"`: float (correct / total), e.g. 0.83
  - `"results"`: list of dicts, each with keys `"question"`, `"expected"`, `"chosen"`, `"correct"`

```python
# Example return value:
# {
#     "accuracy": 0.833,
#     "results": [
#         {"question": "...", "expected": "lookup_employee", "chosen": "lookup_employee", "correct": True},
#         ...
#     ]
# }
```

In [ ]:
def test_tool_selection(descriptions: dict, questions: list, llm) -> dict:
    """Test how accurately the LLM selects tools given a set of descriptions.
    
    Args:
        descriptions: dict mapping tool_name -> description string
        questions: list of dicts with 'question' and 'expected_tool' keys
        llm: the LangChain LLM instance
    
    Returns:
        dict with 'accuracy' (float) and 'results' (list of dicts)
    """
    # ---- TODO: Implement this function ----
    # 1. Build the system prompt from descriptions
    # 2. Loop over each question
    # 3. Send to LLM, parse tool choice
    # 4. Track correct/total and build results list
    # 5. Return {"accuracy": ..., "results": [...]}
    return "___"


# --- Validation ---
todo1_result = test_tool_selection(GOOD_DESCRIPTIONS, TEST_QUESTIONS, llm)

todo1_pass = (
    isinstance(todo1_result, dict)
    and "accuracy" in todo1_result
    and "results" in todo1_result
    and isinstance(todo1_result["accuracy"], float)
    and isinstance(todo1_result["results"], list)
    and len(todo1_result["results"]) == len(TEST_QUESTIONS)
    and all("correct" in r for r in todo1_result["results"])
)

print(f"TODO 1: {'[PASS]' if todo1_pass else '[FAIL]'} test_tool_selection returns correct structure")
if todo1_pass:
    print(f"  Accuracy with good descriptions: {todo1_result['accuracy']:.0%}")
    for r in todo1_result["results"]:
        tag = "OK" if r["correct"] else "MISS"
        print(f"    [{tag}] {r['expected']:20s} -> {str(r['chosen']):20s} | {r['question'][:50]}")
score_todo1 = 4 if todo1_pass else 0

---

## TODO 2: Medium-Quality Descriptions and 3-Level Comparison (3 points)

Create your own **medium-quality** descriptions that are better than the bad ones but not as precise as the good ones.

**Guidelines for medium descriptions:**
- Use real words (not "do stuff") but keep them generic
- Example: instead of `"do stuff"` use `"find information about people"`
- Don't include examples or argument format details

Then run all three levels through `test_tool_selection` and save a comparison report.

In [ ]:
# ---- TODO: Define medium-quality descriptions ----
MEDIUM_DESCRIPTIONS = "___"

# ---- TODO: Run test_tool_selection for all 3 levels ----
# bad_report = test_tool_selection(BAD_DESCRIPTIONS, TEST_QUESTIONS, llm)
# medium_report = test_tool_selection(MEDIUM_DESCRIPTIONS, TEST_QUESTIONS, llm)
# good_report = test_tool_selection(GOOD_DESCRIPTIONS, TEST_QUESTIONS, llm)

# ---- TODO: Save comparison report to file ----
# report_path = os.path.join(OUTPUT_DIR, "description_comparison.txt")
# Write a report showing: level name, accuracy, and per-question results for all 3


# --- Validation ---
todo2_checks = (
    isinstance(MEDIUM_DESCRIPTIONS, dict)
    and len(MEDIUM_DESCRIPTIONS) == 5
    and all(name in MEDIUM_DESCRIPTIONS for name in TOOL_FUNCTIONS)
    and os.path.exists(os.path.join(OUTPUT_DIR, "description_comparison.txt"))
)

print(f"TODO 2: {'[PASS]' if todo2_checks else '[FAIL]'} Medium descriptions + 3-level comparison report")
if todo2_checks:
    report_path = os.path.join(OUTPUT_DIR, "description_comparison.txt")
    with open(report_path) as f:
        print(f"  Report saved to: {report_path}")
        print(f"  Report preview (first 500 chars):")
        print(f"  {f.read()[:500]}")
score_todo2 = 3 if todo2_checks else 0

---

## TODO 3: Disambiguation Experiment (3 points)

Add a **6th tool** `convert_currency(amount_and_pair)` that converts currency.

**The experiment:**
1. Give `convert_currency` an **ambiguous** description that could be confused with `calculate_shipping` (both deal with numbers/money)
2. Test with 2 shipping questions and 2 currency questions
3. Then give `convert_currency` a **clear** description that distinguishes it from shipping
4. Test again and compare

This teaches that descriptions must be **unique and non-overlapping**.

In [ ]:
# The 6th tool implementation (provided)
def convert_currency(amount_and_pair: str) -> str:
    """Convert between currencies. Input: 'AMOUNT FROM TO', e.g., '100 USD INR'."""
    rates = {
        ("usd", "inr"): 83.5,
        ("inr", "usd"): 0.012,
        ("eur", "inr"): 90.2,
        ("inr", "eur"): 0.011,
        ("usd", "eur"): 0.92,
        ("eur", "usd"): 1.09,
    }
    parts = amount_and_pair.strip().lower().split()
    try:
        amount = float(parts[0])
        from_curr = parts[1]
        to_curr = parts[2]
        rate = rates.get((from_curr, to_curr))
        if rate:
            return f"{amount} {from_curr.upper()} = {amount * rate:.2f} {to_curr.upper()}"
        return f"Rate not found for {from_curr.upper()} -> {to_curr.upper()}"
    except (ValueError, IndexError):
        return f"Invalid input: '{amount_and_pair}'. Use format: '100 USD INR'"


# Add to global function map
TOOL_FUNCTIONS["convert_currency"] = convert_currency

# Test questions specifically designed to test shipping vs currency confusion
DISAMBIGUATION_QUESTIONS = [
    {"question": "How much does it cost to ship a 5 kg box?", "expected_tool": "calculate_shipping"},
    {"question": "What is 200 USD in Indian Rupees?", "expected_tool": "convert_currency"},
    {"question": "Calculate the delivery charge for a 1.2 kg parcel", "expected_tool": "calculate_shipping"},
    {"question": "Convert 5000 INR to Euros", "expected_tool": "convert_currency"},
]

# ---- TODO: Create AMBIGUOUS descriptions for all 6 tools ----
# Make convert_currency's description easily confused with calculate_shipping
# Example ambiguous description: "calculate money amounts"
AMBIGUOUS_DESCS = "___"

# ---- TODO: Create CLEAR descriptions for all 6 tools ----
# Make convert_currency clearly distinct from calculate_shipping
CLEAR_DESCS = "___"

# ---- TODO: Run tests and compare ----
# ambiguous_report = test_tool_selection(AMBIGUOUS_DESCS, DISAMBIGUATION_QUESTIONS, llm)
# clear_report = test_tool_selection(CLEAR_DESCS, DISAMBIGUATION_QUESTIONS, llm)
# Print comparison showing ambiguous vs clear accuracy


# --- Validation ---
todo3_checks = (
    isinstance(AMBIGUOUS_DESCS, dict)
    and isinstance(CLEAR_DESCS, dict)
    and "convert_currency" in AMBIGUOUS_DESCS
    and "convert_currency" in CLEAR_DESCS
    and len(AMBIGUOUS_DESCS) == 6
    and len(CLEAR_DESCS) == 6
)

print(f"TODO 3: {'[PASS]' if todo3_checks else '[FAIL]'} Disambiguation experiment with 6 tools")
score_todo3 = 3 if todo3_checks else 0

---

## Scoring Summary

In [ ]:
total = score_todo1 + score_todo2 + score_todo3
print("=" * 50)
print("LAB 05: TOOL DESCRIPTION ENGINEERING - SCORING")
print("=" * 50)
print(f"TODO 1 (test_tool_selection function):   {score_todo1}/4")
print(f"TODO 2 (medium descriptions + report):   {score_todo2}/3")
print(f"TODO 3 (disambiguation experiment):      {score_todo3}/3")
print(f"{'─' * 50}")
print(f"Total: {total}/10")
print("=" * 50)

## Key Takeaways

- **Tool descriptions are the interface** between the LLM and your tools. The LLM cannot read your code -- it only sees descriptions.
- **Vague descriptions** ("do stuff", "handle data") cause frequent misrouting.
- **Good descriptions** include: what the tool does, what input it expects, and an example.
- **Overlapping descriptions** between tools cause confusion. Each tool must have a **unique, distinguishable** purpose in its description.
- This is a form of **prompt engineering** applied specifically to tool definitions -- a critical production skill.